## "Quiz Master"

This is my idea for a "visible agent loop":

- The user tells the system on what topic they wants questions to be asked and also the number of questions
- The agent first prepares a list of questions behind the scenes
- The system asks the questions one by one - lets make it multiple choice questions with one correct answer
- As soon as the user answers them, the system prints a status - a green tick for correct answer, a red cross for wrong ones and a dash for remaining questions

In [25]:
from rich.console import Console
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from agents import Agent, WebSearchTool, trace, Runner, function_tool
import os
from agents import set_tracing_export_api_key
import gradio as gr


In [26]:
load_dotenv(override=True)
MODEL_NAME = "gpt-5.4-mini"
NUM_QUESTIONS = 10

In [27]:
os.environ["OPENAI_LOG"] = "debug"
set_tracing_export_api_key(os.getenv("OPENAI_TRACING_KEY"))

In [28]:
class Choice(BaseModel):
    optionId: str = Field(..., description="The unique identifier for the choice option. Foe example, 'A', 'B', 'C', etc.")
    text: str = Field(..., description="The text of the choice option.")

class QuestionAndAnswer(BaseModel):
    question: str = Field(..., description="The question to be answered.")
    choices: list[Choice] = Field(..., description="The list of choice options for the question.")
    answer: str = Field(..., description="The correct answer to the question, represented by the optionId of the correct choice.")

class QuestionsAndAnswers(BaseModel):
    questions: list[QuestionAndAnswer] = Field(..., description="A list of questions and their corresponding answers.")

In [29]:
INSTRUCTIONS = """
You are a quiz master. Generate a set of multiple-choice questions with four options each. Each question should have one correct answer. The questions should be clear and concise, and the options should be plausible to make the quiz challenging.
"""
quiz_master_agent = Agent(
    name="QuizMaster",
    instructions=INSTRUCTIONS,
    output_type=QuestionsAndAnswers,
    model=MODEL_NAME
    )

In [30]:
task = f"Generate {NUM_QUESTIONS} multiple-choice questions on the topic of Python programming."
result = await Runner.run(quiz_master_agent, task)

In [31]:
question_answers = result.final_output
print(question_answers)

questions=[QuestionAndAnswer(question='What is the output of len([1, 2, 3])?', choices=[Choice(optionId='A', text='2'), Choice(optionId='B', text='3'), Choice(optionId='C', text='4'), Choice(optionId='D', text='Error')], answer='B'), QuestionAndAnswer(question='Which keyword is used to define a function in Python?', choices=[Choice(optionId='A', text='func'), Choice(optionId='B', text='define'), Choice(optionId='C', text='def'), Choice(optionId='D', text='lambda')], answer='C'), QuestionAndAnswer(question='Which of the following is a mutable data type?', choices=[Choice(optionId='A', text='tuple'), Choice(optionId='B', text='string'), Choice(optionId='C', text='list'), Choice(optionId='D', text='int')], answer='C'), QuestionAndAnswer(question='What does the append() method do for a list?', choices=[Choice(optionId='A', text='Adds an item to the end of the list'), Choice(optionId='B', text='Removes the last item'), Choice(optionId='C', text='Sorts the list'), Choice(optionId='D', text='

In [ ]:
# 0 for wrong answer, 1 for correct answer
answers = []

In [24]:
def check_answer(question: QuestionAndAnswer, user_answer: str) -> bool:
    """
    Check if the user's answer is correct for the given question.

    Args:
        question (QuestionAndAnswer): The question to check against.
        user_answer (str): The user's answer, represented by the optionId.

    Returns:
        bool: True if the answer is correct, False otherwise.
    """
    
    return question.answer == user_answer

In [3]:
# generate questions
questions = [
    {"question": "What is the capital of France?", "answer": "Paris"},
    {"question": "Who wrote 'Romeo and Juliet'?", "answer": "William Shakespeare"},
    {"question": "What is the largest planet in our solar system?", "answer": "Jupiter"}
]


In [ ]:
@function_tool
def record_answer(question_index: int, user_answer: str):
    """
    Record the user's answer for a specific question.

    Args:
        question_index (int): The index of the question in the questions list.
        user_answer (str): The user's answer, represented by the optionId.
    """
    if 0 <= question_index < len(questions):
        is_correct = check_answer(questions[question_index], user_answer)
        answers[question_index] = is_correct
    else:
        print(f"Invalid question index: {question_index}")

In [ ]:
@function_tool
def print_status() -> str:
    result = ""
    for index in range(NUM_QUESTIONS):
        if index >= len(answers):
            result += "[bold yellow]-[/bold yellow] "
        elif answers[index] == True:
            result += "[bold green]✓[/bold green] "
        elif answers[index] == False:
            result += "[bold red]✗[/bold red] "
        else:
            result += "[bold yellow]-[/bold yellow] "
    return result

In [14]:
status = print_status()
Console().print(status)

✗ ✓ -

In [ ]:
INSTRUCTIONS = f"""
You are a quiz master bot.

You'll be given a set of multiple-choice questions with four options each. Each question has one correct answer. 
Your task is to present the questions to the user, collect their answers, and provide feedback on whether their answers are correct or incorrect.

The questions will be provided in a structured format, and you should ensure that the user understands how to respond with their chosen option (e.g., 'A', 'B', 'C', or 'D').

You will ask the questions one by one, and answer would be fed back to you for each of them.
You will then call another tool record_answer to record the answer and its correctness. 
After that call print_status tool to get the current status of the quiz and display it to the user.

The question-choices-answer list is below in JSON format. You will use this to present the questions to the user and check their answers.
{question_answers}
"""

quiz_master_bot = Agent(name="QuizMasterBot", instructions=INSTRUCTIONS, model=MODEL_NAME, tools=[record_answer, print_status])

In [ ]:
last_response = None

In [ ]:
async def async_chat(message, history):
    global last_response
    if last_response is None:
        # first turn - noting to carry forward
        agent_input = message
    else:
        # subsequent turns - carry forward the last response
        agent_input = last_response.to_input_list() + [{"role": "user", "content": message}]

    last_response = await Runner.run(quiz_master_bot, agent_input)
    return last_response.final_output
    

In [ ]:
gr.ChatInterface(async_chat).launch(inbrowser=True)
